# TT-SVD: 3階テンソルでの基礎実験

このNotebookでは、添付チャットで学習済みの **TT定義・TT-rank・unfolding・TT-SVD** を、
小さい3階テンソルで数値的に確認する。

新しい理論は扱わない。まずはライブラリ化せず、shapeを追いながら自分で実装する。

数値誤差を理論上の等式と区別しやすくするため、この基礎実験では `float64` を使う。

## ゴール

- $X \in \mathbb{R}^{n_1\times n_2\times n_3}$ を2回のSVDで3個のTT coreへ分解する
- core shapeを確認する
- 打ち切りなしなら元テンソルを数値誤差の範囲で再構成できることを確認する
- SVDの出力shapeではなく、unfoldingの数値rankをTT-rankとして扱う

> このNotebookでは gauge freedom 以降は扱わない。

## 1. 実験対象テンソル

$X.shape=(2,3,4)$ の小さいテンソルを使う。

### やること

- seedを固定して `X` を作る
- `shape`, `ndim`, `numel()` を確認する

### ヒント

- `torch.manual_seed`
- `torch.randn`
- `torch.set_default_dtype(torch.float64)`

打ち切りなしの再構成誤差は、`float64` ならおおむね $10^{-14}$〜$10^{-15}$ 程度まで下がり得る。

In [ ]:
import torch

torch.set_default_dtype(torch.float64)
torch.manual_seed(0)

# TODO: shape=(2, 3, 4) のテンソル X を作る
X = None

# TODO: shape / ndim / numel() を表示する

## 2. 第1切断のunfolding

最初の切断は $i_1 \mid i_2 i_3$ なので、行列shapeは $(n_1,\ n_2 n_3)$ になる。

### やること

- `X` を第1切断で2次元行列へreshapeする
- shapeが `(2, 12)` になることを確認する

### ヒント

この段階では値の並べ替えは不要。先頭軸と残りの軸をまとめるだけ。

In [ ]:
# TODO: X1 を第1切断のunfolding行列へ変形する
X1 = None

# TODO: shape=(2, 12) を確認する

## 3. 第1回SVDと第1core

$X^{\langle 1\rangle}=U_1\Sigma_1V_1^\top$ を計算する。打ち切りはまだ行わない。

### やること

1. SVDする
2. $r_1=\operatorname{rank}(X^{\langle1\rangle})$ を求める
3. `U1`, `S1`, `Vh1` を `r1` まで切る
4. `U1` を `(1, n1, r1)` へreshapeして `G1` を作る
5. 残りの $\Sigma_1V_1^\top$ を作る

### ヒント

- `torch.linalg.svd(..., full_matrices=False)`
- `r1 = int(torch.linalg.matrix_rank(X1).item())`
- `U1 = U1[:, :r1]`, `S1 = S1[:r1]`, `Vh1 = Vh1[:r1, :]`
- `torch.diag`

> `torch.linalg.matrix_rank` は、しきい値に基づく**数値rank**を返す。今回のランダムな小テンソルでは問題になりにくいが、ほぼゼロの特異値を持つテンソルでは tolerance の選び方が結果に影響する。

`full_matrices=False` が返す列数そのものとTT-rankを混同しないこと。TT-rankとして使うのは数値的に非零と判定された特異方向の数。

In [ ]:
# TODO: X1 をSVDする
U1 = None
S1 = None
Vh1 = None

# TODO: r1 = matrix_rank(X1) とし、U1 / S1 / Vh1 を r1 まで切る
r1 = None

# TODO: G1.shape = (1, n1, r1)
G1 = None

# TODO: 次のステップへ渡す B1 = Sigma1 @ Vh1
B1 = None

## 4. 第2回SVDと残りのcore

`B1` は概念的に $(r_1,n_2,n_3)$ へ戻せる。
次のSVDでは $(r_1n_2)\mid n_3$ とまとめる。

### やること

- `B1` を `(r1*n2, n3)` にreshape
- SVD
- $r_2=\operatorname{rank}(B_2)$ を求める
- `U2`, `S2`, `Vh2` を `r2` まで切る
- `G2.shape = (r1, n2, r2)`
- `G3.shape = (r2, n3, 1)`

### ヒント

- `r2 = int(torch.linalg.matrix_rank(B2).item())`
- `U2 = U2[:, :r2]`, `S2 = S2[:r2]`, `Vh2 = Vh2[:r2, :]`
- 最後のcoreには右端rank `1` を明示的に追加する。

In [ ]:
# TODO: B1 を第2回SVD用の行列 B2 にreshapeする
B2 = None

# TODO: B2 をSVDする
U2 = None
S2 = None
Vh2 = None

# TODO: r2 = matrix_rank(B2) とし、U2 / S2 / Vh2 を r2 まで切る
r2 = None
G2 = None
G3 = None

# TODO: 3つのcore shapeを表示する

## 5. TT coreから再構成

3個のcoreの内部rankを縮約して $\hat X$ を作る。

### やること

- `G1`, `G2`, `G3` を縮約する
- 境界rank 1を取り除き、`X_hat.shape == X.shape` にする
- relative Frobenius error $\|X-\hat X\|_F/\|X\|_F$ を求める

### ヒント

- `torch.tensordot` または `torch.einsum`
- 縮約する軸は、左coreの右rankと右coreの左rank
- `torch.linalg.vector_norm`

`float64` で打ち切りなしなら、誤差が $10^{-14}$〜$10^{-15}$ 程度でも異常ではなく、浮動小数点演算による数値誤差と解釈できる。

In [ ]:
# TODO: G1, G2, G3 から X_hat を再構成する
X_hat = None

# TODO: relative Frobenius error を計算する
rel_error = None

# TODO: shapeと誤差を表示する

## 6. 確認

- `G1.shape == (1, 2, r1)`
- `G2.shape == (r1, 3, r2)`
- `G3.shape == (r2, 4, 1)`
- 打ち切りなしなら相対誤差は数値誤差の範囲
- `r1`, `r2` は各SVD行列の数値rankであり、各切断のTT-rankに対応する

次は一般$d$階へ拡張し、TT-rankと元テンソルのunfolding rankを直接照合する。